# Experiment 6: Forward Propagation and Backpropagation

## Aim
To implement Artificial Neural Network training using Forward Propagation and Backpropagation.

## Objective
To understand how neural networks compute output and update weights using error correction.

## Theory

### Forward Propagation
Data flows from input → hidden → output layer.

**Linear Combination:**
$$Z = W \cdot X + b$$

**Activation:**
$$A = \sigma(Z)$$

Where:
- Z = weighted sum of inputs plus bias
- A = activated output (after sigmoid function)
- σ = sigmoid activation function

### Backpropagation
Weights are updated using error through gradient descent:

$$W = W - \eta \cdot \frac{\partial L}{\partial W}$$

Where:
- η (eta) = learning rate (step size)
- L = loss function (error)
- ∂L/∂W = gradient (how much weight contributed to error)

We use **Sigmoid activation** and **Mean Squared Error (MSE)**.

**Key Insight:** The gradient tells us how sensitive the loss is to each weight. We move weights in the opposite direction of the gradient to reduce error.


In [8]:

import numpy as np


## Key Definitions

### Forward Propagation
The process of passing input data through the network layers to compute the output. Data flows from:
- **Input Layer** → **Hidden Layer** → **Output Layer**

At each layer, we compute: Z = W·X + b, then apply activation A = σ(Z)

### Backpropagation
An algorithm that computes how much each weight contributed to the error, then updates weights to reduce that error. It works backwards from output to input layer.

### Sigmoid Activation Function
A smooth function that squashes values between 0 and 1:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Why use it?** 
- Converts linear combinations into non-linear outputs (enables learning complex patterns)
- Output is bounded between 0-1 (probability-like interpretation)
- Smooth and differentiable (needed for backpropagation)

### Loss Function (MSE)
Measures how far our prediction is from the true value:
$$L = \frac{1}{n} \sum (y_{true} - y_{pred})^2$$

Lower loss = better predictions

### Gradient Descent
An optimization algorithm that updates weights in the direction that reduces loss:
$$W = W - \eta \cdot \frac{\partial L}{\partial W}$$

Where η (eta) is the **learning rate** (step size).


## Step 1: Define Activation Functions

We need two functions for the sigmoid activation:

1. **sigmoid(x)**: Converts any value to a range between 0 and 1
2. **sigmoid_derivative(x)**: The slope of sigmoid at a given point (used in backpropagation)

**The Derivative:**
$$\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$$

This derivative is crucial because it tells us how sensitive the output is to changes in input, which determines how much we adjust the weights.


In [9]:

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)


In [10]:

# Simple dataset (from practical idea)
X = np.array([[1, 0, 1]])
y = np.array([[1]])


## Step 2: Prepare the Dataset

We define a simple training dataset:

**Input (X)**: [1, 0, 1] (3-dimensional input)
- This is a single sample with 3 features/attributes

**Output (y)**: [1] (1-dimensional output)
- The target value we want the network to predict
- In this case, we want the network to learn that input [1, 0, 1] should map to output 1

**Network Architecture:**
```
Input Layer (3 neurons)
    ↓
Hidden Layer (2 neurons)
    ↓
Output Layer (1 neuron)
    ↓
Prediction (0 or 1)
```

This is a simple 3-2-1 neural network.


In [11]:

np.random.seed(42)

# Initialize weights
W1 = np.random.randn(3, 2)
W2 = np.random.randn(2, 1)

# Bias
b1 = np.zeros((1, 2))
b2 = np.zeros((1, 1))

learning_rate = 0.1
epochs = 1000


## Step 3: Initialize Weights and Hyperparameters

### Weight Initialization

**W1** (3×2 matrix): Weights connecting Input Layer to Hidden Layer
- 3 inputs × 2 hidden neurons = 6 weights

**W2** (2×1 matrix): Weights connecting Hidden Layer to Output Layer
- 2 hidden neurons × 1 output neuron = 2 weights

Weights are initialized randomly with `np.random.randn()` to break symmetry (if all weights started the same, neurons would learn identical patterns).

### Bias Terms

**b1** and **b2**: Bias vectors for hidden and output layers
- Allow neurons to shift their activation curves
- Initialized to zero (will be learned during training)

### Hyperparameters

**Learning Rate (η = 0.1):**
- Controls how much we adjust weights per step
- Too high: weights oscillate, may not converge
- Too low: training is very slow
- 0.1 is a reasonable starting value

**Epochs = 1000:**
- Number of times the network trains on the entire dataset
- Each epoch updates weights based on the error


In [12]:

for epoch in range(epochs):

    # ---- Forward Propagation ----
    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)

    # ---- Loss ----
    loss = np.mean((y - A2)**2)

    # ---- Backpropagation ----
    dA2 = (A2 - y)
    dZ2 = dA2 * sigmoid_derivative(A2)
    dW2 = np.dot(A1.T, dZ2)
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * sigmoid_derivative(A1)
    dW1 = np.dot(X.T, dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    # ---- Update ----
    W1 -= learning_rate * dW1
    W2 -= learning_rate * dW2
    b1 -= learning_rate * db1
    b2 -= learning_rate * db2

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss}")


Epoch 0, Loss: 0.053108332069688666
Epoch 100, Loss: 0.020834707432048842
Epoch 200, Loss: 0.012033724825635172
Epoch 300, Loss: 0.008236487152476017
Epoch 400, Loss: 0.006180821726801715
Epoch 500, Loss: 0.004910430960158512
Epoch 600, Loss: 0.004054631169262629
Epoch 700, Loss: 0.003442194730622751
Epoch 800, Loss: 0.0029839028313703377
Epoch 900, Loss: 0.0026289947424799113


## Step 4: Training Loop - Forward Propagation and Backpropagation

This is where the network learns!

### Forward Propagation (Computing Output)

**Layer 1 (Input → Hidden):**
- Z1 = X · W1 + b1 (linear combination)
- A1 = sigmoid(Z1) (apply activation)

**Layer 2 (Hidden → Output):**
- Z2 = A1 · W2 + b2 (linear combination)
- A2 = sigmoid(Z2) (apply activation - this is our prediction!)

### Loss Calculation

Loss = Mean Squared Error = (y_true - y_pred)²

This tells us how wrong our prediction was.

### Backpropagation (Computing Gradients)

Working **backwards** from output to input:

**Output Layer Gradient:**
- dA2 = (A2 - y) → how much output differs from target
- dZ2 = dA2 × sigmoid_derivative(A2) → sensitivity of Z2 to changes
- dW2 = A1ᵀ · dZ2 → how much W2 should change
- db2 = sum(dZ2) → how much b2 should change

**Hidden Layer Gradient:**
- dA1 = dZ2 · W2ᵀ → propagate error backwards
- dZ1 = dA1 × sigmoid_derivative(A1) → sensitivity of Z1
- dW1 = Xᵀ · dZ1 → how much W1 should change
- db1 = sum(dZ1) → how much b1 should change

### Weight Update (Gradient Descent)

Subtract the gradient times learning rate from current weights:
- W1 = W1 - learning_rate × dW1
- W2 = W2 - learning_rate × dW2
- b1 = b1 - learning_rate × db1
- b2 = b2 - learning_rate × db2

This pushes weights in the direction that reduces loss!


In [13]:

# Final prediction
Z1 = np.dot(X, W1) + b1
A1 = sigmoid(Z1)

Z2 = np.dot(A1, W2) + b2
A2 = sigmoid(Z2)

print("Final Output:", A2)


Final Output: [[0.95155841]]


## Step 5: Make Final Prediction

After training for 1000 epochs, the network has learned the weights and biases.

Now we perform **one final forward propagation** using the trained weights to get our prediction.

**Expected Result:**
- Input: [1, 0, 1]
- Output: Should be close to 1 (our target)

The output value will be between 0 and 1 (due to sigmoid activation):
- If output ≈ 0.9 or higher: Network predicts class 1 ✓
- If output ≈ 0.1 or lower: Network predicts class 0 ✗

A value close to 1 means the network successfully learned the pattern!


## Summary & Key Takeaways

### What We Accomplished

1. **Forward Propagation**: Computed predictions by passing input through weights and activation functions
2. **Loss Calculation**: Measured how far our predictions were from true values
3. **Backpropagation**: Computed gradients showing which weights caused the error
4. **Weight Update**: Adjusted weights using gradient descent to minimize loss

### The Complete Training Process

```
Initialize Weights Randomly
    ↓
For each epoch:
    1. Forward Propagation → Compute prediction
    2. Calculate Loss → Measure error
    3. Backpropagation → Calculate gradients
    4. Update Weights → Reduce error
    ↓
After many epochs: Network has learned the pattern
```

### Why This Matters

- **Forward Propagation** enables the network to make predictions
- **Backpropagation** enables the network to learn from errors
- Together, they form the foundation of deep learning

### Real-World Applications

This technique is used in:
- **Image Recognition**: Networks learn to identify objects
- **Natural Language Processing**: Models learn to understand and generate text
- **Medical Diagnosis**: Networks trained to detect diseases from X-rays
- **Stock Prediction**: Models trained to forecast market trends

### Conclusion

We successfully implemented a complete neural network training pipeline using forward propagation to compute outputs and backpropagation to optimize weights through error correction. This is the fundamental algorithm powering modern artificial intelligence!
